In [15]:
import ollama
import pandas as pd
from yahoo_finance import MarketData
from news_embedder import NewsEmbedder

OLLAMA_HOST = "http://100.68.136.71:11434"

# Verifica se o modelo já está disponível antes de puxar
client = ollama.Client(host=OLLAMA_HOST)


# Etapa 2 — Engenharia de Features

Este notebook combina dados de mercado (Yahoo Finance) com embeddings de notícias (Ollama) em um dataset unificado para treinamento.

**Fontes de dados:**
- `yahoo_finance.py` → `MarketData.features()`: Close, Volume, retorno diário, médias móveis 7/21 dias, volatilidade 21 dias, lags 1-5
- `news_embedder.py` → `NewsEmbedder`: embeddings de 1024 dimensões via modelo `qwen3-embedding:4b`

**Agregação e alinhamento:**
- Notícias do mesmo dia são agregadas via média ponderada por recência (`_weighted_mean`)
- Embeddings são alinhados com preços via `merge_with_prices()` (left join + forward-fill para dias sem notícias)

**Dimensões finais do dataset:** 230 dias × 1.035 features (8 price + 1.024 embeddings + 3 PCA auxiliares)

**Saída:** `dataset_full.csv` — dataset completo pronto para treinamento

In [16]:
client.list().models

13:29:27 [INFO] HTTP Request: GET http://100.68.136.71:11434/api/tags "HTTP/1.1 200 OK"


[Model(model='qwen3-embedding:4b', modified_at=datetime.datetime(2026, 3, 16, 11, 3, 45, 563525, tzinfo=TzInfo(-10800)), digest='df5bd2e3c74cd8d069d21dc038f1b359fcdc9458fce1c99bd43c9eb1518ff907', size=2496704041, details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='4.0B', quantization_level='Q4_K_M')),
 Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 16, 31, 728797, tzinfo=TzInfo(-10800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')),
 Model(model='lfm2.5-thinking:latest', modified_at=datetime.datetime(2026, 3, 2, 20, 12, 22, 506453, tzinfo=TzInfo(-10800)), digest='95bd9d45385f33bfe96d8b3651c8569e152f21f5bdb7c19894ffde650e9cf140', size=731163903, details=ModelDetails(parent_model='', format='gguf', family='lfm2', familie

In [23]:
embedding_model = "qwen3-embedding:0.6b"

In [24]:
modelos = [m.model for m in client.list().models]
if embedding_model not in modelos:
    print(f"Baixando {embedding_model}...")
    client.pull(embedding_model)

13:30:28 [INFO] HTTP Request: GET http://100.68.136.71:11434/api/tags "HTTP/1.1 200 OK"


In [ ]:

# Notícias
df_news = pd.read_json("../1.news/itub4_noticias.json", convert_dates=False)
articles  = df_news.to_dict(orient="records")

# Preços — período estendido para 5 anos (alinhado com dados de notícias desde 2021)
md      = MarketData("ITUB4.SA")
X_price = md.features(period="5y", lags=5)
y       = md.target(period="5y", horizon=5)

# Embedder
ne = NewsEmbedder(
    model=embedding_model,
    fields=["title", "excerpt", "content"],
    ollama_host=OLLAMA_HOST,
    cache_path="embeddings_cache.npz",
    summarizer_model='lfm2.5-thinking'
)


In [29]:
df_news

,id,date,modified,title,link,excerpt,content,author_id,author_name,hat,categories,tags,keywords
0,3252277,2026-03-16T13:08:54,2026-03-16T13:09:01,Cosan (CSAN3): o que esperar da holding após r...,https://www.infomoney.com.br/mercados/cosan-cs...,Cosan vem passando por um processo de reestrut...,Após a Cosan (CSAN3) divulgar seus resultados ...,54205,Felipe Moreira,De olho em CSAN3,[1],"[332, 2691, 629, 210752, 381]","[Ações, Cosan, Goldman Sachs, Recomendação neu..."
1,3251628,2026-03-16T10:56:56,2026-03-16T10:57:01,"Com maior peso de estatais na AL, Brasil abre ...",https://www.infomoney.com.br/mercados/com-maio...,Banco atualizou sua cesta de ações de estatais...,"De olho na proximidade do pleito de outubro, o...",25,Lara Rizério,Cenário para eleições,[1],"[288694, 1172, 1414, 1440, 1880, 12, 2517]","[Axia, Bradesco BBI, Embraer, Estatais, MSCI, ..."
2,3250267,2026-03-16T06:04:39,2026-03-16T06:06:01,"Prévia do PIB, indústria nos EUA, conflito no ...",https://www.infomoney.com.br/mercados/previa-p...,InfoMoney reúne as principais informações que ...,A sessão desta segunda-feira (16) atualiza mai...,620517,"Erick Souza, Felipe Moreira",O que vai mexer com a Bolsa,[1],"[986, 382, 1383, 283800, 539, 192609, 194340, ...","[5 Assuntos, Banco Central, Donald Trump, Elei..."
3,3250055,2026-03-13T17:51:27,2026-03-13T18:19:20,Ibovespa fecha abaixo de 178 mil pontos sem al...,https://www.infomoney.com.br/mercados/ibovespa...,O volume financeiro nesta sexta-feira somou R$...,"SÃO PAULO, 13 Mar (Reuters) – O Ibovespa fecho...",43,Reuters,Mercado doméstico,[1],"[653, 1649, 2314]","[Ibovespa, Irã, Trader]"
4,3250081,2026-03-13T18:13:47,2026-03-13T18:13:49,"Previ recebeu R$ 4,4 bi em dividendos e JCP e ...",https://www.infomoney.com.br/onde-investir/pre...,Dividendos e juros sobre capital próprio ajuda...,"A Previ recebeu R$ 4,4 bilhões em dividendos e...",44,Estadão Conteúdo,Dividendos e JCP,[2476],"[527, 1695, 814]","[Dividendos, JCP, Previ]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2567,1626362,2021-04-20T08:20:50,2021-04-20T09:19:21,Dados de produção da Vale e de vendas do Carre...,https://www.infomoney.com.br/mercados/dados-de...,Confira os destaques do noticiário corporativo...,SÃO PAULO – O noticiário corporativo desta ter...,293,Equipe InfoMoney,Radar InfoMoney,[1],"[332, 1227, 2803, 2068, 2517]","[Ações, Carrefour, Linx, Radar InfoMoney, Vale]"
2568,1625265,2021-04-16T12:23:51,2021-04-16T12:25:03,B3 mantém Locaweb (LWSA3) e inclui units do Ba...,https://www.infomoney.com.br/mercados/b3-mante...,"Com isso, caso confirmada na próxima prévia, a...",A B3 divulgou nesta sexta-feira (16) a segunda...,293,Equipe InfoMoney,Nova carteira,[1],"[332, 2840, 653, 1742]","[Ações, Banco Inter, Ibovespa, Locaweb]"
2569,1624516,2021-04-15T17:44:23,2021-04-15T17:45:41,"Ações de Vale e frigoríficos voltam a subir, P...",https://www.infomoney.com.br/mercados/acoes-de...,Confira os destaques da B3 na sessão desta qui...,SÃO PAULO – O grande destaque do mercado nesta...,25,Lara Rizério,Destaques da Bolsa,[1],"[332, 2782, 2445, 2614, 2495, 2712, 2490]","[Ações, Arezzo, Bradesco, Cia Hering, Itaú Uni..."
2570,1623847,2021-04-14T17:40:46,2021-04-14T18:01:26,"Ações de Vale, siderúrgicas, Petrobras e frigo...",https://www.infomoney.com.br/mercados/acoes-da...,Confira os destaques da B3 na sessão desta qua...,SÃO PAULO – O destaque do Ibovespa na sessão d...,25,Lara Rizério,Destaques da Bolsa,[1],"[332, 2550, 2687, 2827, 2713, 2517]","[Ações, Americanas, B2W Digital, CVC Brasil, J..."


In [30]:
# Dataset final
X_full = ne.merge_with_prices(X_price, articles)
df     = X_full.join(y).dropna()
X      = df.drop(columns=["target"])
y      = df["target"]

print(f"Shape final: {X.shape}")  # (dias, price_features + 768 emb dims)

13:33:24 [INFO] Iniciando embedding de 2572 artigos com 'qwen3-embedding:0.6b'
13:33:25 [INFO] HTTP Request: POST http://100.68.136.71:11434/api/embed "HTTP/1.1 200 OK"
13:33:25 [INFO]   [   1/2572] 2026-03-16 | embed |   0.4s | ETA 17m36s | chars=6562
13:33:25 [INFO] HTTP Request: POST http://100.68.136.71:11434/api/embed "HTTP/1.1 200 OK"
13:33:25 [INFO]   [   2/2572] 2026-03-16 | embed |   0.3s | ETA 14m46s | chars=5344
13:33:25 [INFO] HTTP Request: POST http://100.68.136.71:11434/api/embed "HTTP/1.1 200 OK"
13:33:25 [INFO]   [   3/2572] 2026-03-16 | embed |   0.3s | ETA 14m27s | chars=5748
13:33:26 [INFO] HTTP Request: POST http://100.68.136.71:11434/api/embed "HTTP/1.1 200 OK"
13:33:26 [INFO]   [   4/2572] 2026-03-13 | embed |   0.3s | ETA 14m18s | chars=5408
13:33:26 [INFO] HTTP Request: POST http://100.68.136.71:11434/api/embed "HTTP/1.1 200 OK"
13:33:26 [INFO]   [   5/2572] 2026-03-13 | embed |   0.1s | ETA 12m36s | chars=1048
13:33:27 [INFO] HTTP Request: POST http://100.68.13

Shape final: (225, 1035)


In [31]:
X_full

,Close,Volume,return,ma7,ma21,std21,lag_1,lag_2,lag_3,lag_4,...,emb_1014,emb_1015,emb_1016,emb_1017,emb_1018,emb_1019,emb_1020,emb_1021,emb_1022,emb_1023
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-14,28.989862,27027200,0.016347,28.325033,28.552147,0.372174,28.523584,28.380116,28.496687,27.985571,...,0.031040,-0.002682,-0.000366,-0.041628,-0.027340,0.006508,0.001079,0.026077,0.000236,-0.036171
2025-04-15,29.357506,25716216,0.012682,28.529990,28.579081,0.409033,28.989862,28.523584,28.380116,28.496687,...,0.031142,0.001516,-0.003609,-0.030854,-0.026495,0.009638,-0.025911,-0.011695,0.016893,-0.050896
2025-04-16,29.312668,32431301,-0.001527,28.720856,28.596509,0.432597,29.357506,28.989862,28.523584,28.380116,...,0.040072,-0.000547,0.011218,-0.001519,-0.012881,-0.009819,0.004554,0.018012,0.017389,-0.034635
2025-04-17,29.366468,20502356,0.001835,28.918127,28.613084,0.455626,29.312668,29.357506,28.989862,28.523584,...,0.018080,-0.003437,-0.018135,-0.022424,-0.018967,-0.006076,0.014465,-0.001867,0.029521,-0.065849
2025-04-22,29.940351,21624747,0.019542,29.124365,28.669790,0.539787,29.366468,29.312668,29.357506,28.989862,...,0.007268,-0.018829,-0.015511,-0.005094,-0.007859,0.000201,-0.007234,-0.005571,-0.010946,-0.023275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-10,43.799999,50383100,0.014829,44.101428,46.711363,2.073892,43.160000,42.930000,43.509998,45.009998,...,0.016510,0.004368,-0.024688,-0.041406,0.005131,0.007449,-0.005965,-0.003189,0.005549,-0.068453
2026-03-11,43.889999,18482600,0.002055,43.811428,46.576037,2.163281,43.799999,43.160000,42.930000,43.509998,...,0.031070,0.018682,0.002019,-0.039785,-0.017665,0.000604,0.010232,-0.033142,0.020925,-0.046223
2026-03-12,42.689999,27095200,-0.027341,43.569999,46.309312,2.283212,43.889999,43.799999,43.160000,42.930000,...,0.031070,0.018682,0.002019,-0.039785,-0.017665,0.000604,0.010232,-0.033142,0.020925,-0.046223


In [32]:
X_full.to_csv("dataset_full.csv", index=True)